# Descriptor-family comparison

**Purpose:** Compare the complete, phase and spread descriptor families using grouped Ridge regression.

**Required inputs:** Prepared Baček HDF5; 04 foundational results ZIP.

**Outputs:** Descriptor-family predictions and metrics.

**Usage:** Run after 04. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, fixed experiment definitions and small helpers
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE_ZIP = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
OUT = Path('08_Bacek_Descriptor_Family_Comparison_results')
ARCHIVE = Path('08_Bacek_Descriptor_Family_Comparison_results.zip')
SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
REFERENCE_ZIP_SHA256 = 'd0d5fb49f3f767dc1b99c790afda9aad630bfae5199fced5c5512c14329c7752'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 55)
FOLD_OF = {person: fold for fold, people in OUTER_FOLDS.items() for person in people}
FAMILIES = ('D-PHASE', 'D-SPREAD', 'D-FULL')
NEW_FAMILIES = ('D-PHASE', 'D-SPREAD')
FEATURE_INDICES = {
    'D-PHASE': tuple(6*j + k for j in range(9) for k in range(4)),
    'D-SPREAD': tuple(6*j + k for j in range(9) for k in (4, 5)),
    'D-FULL': tuple(range(54)),
}


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def from_zip_csv(archive, name):
    with archive.open(name) as handle:
        return list(csv.DictReader(io.TextIOWrapper(handle, encoding='utf-8')))


def save_csv(path, records, columns):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)


def rmse(target, prediction):
    err = np.asarray(prediction, dtype=np.float64) - np.asarray(target, dtype=np.float64)
    return float(np.sqrt(np.mean(err**2, dtype=np.float64)))


def metrics(target, prediction):
    target = np.asarray(target, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    err = prediction - target
    sst = float(np.sum((target - target.mean())**2, dtype=np.float64))
    return {
        'n_trials': len(target),
        'RMSE_W_per_kg': rmse(target, prediction),
        'MAE_W_per_kg': float(np.mean(np.abs(err), dtype=np.float64)),
        'bias_W_per_kg': float(np.mean(err, dtype=np.float64)),
        'R2': float(1 - np.sum(err**2, dtype=np.float64)/sst) if sst > 0 else None,
    }

print('Python:', platform.python_version(), '| NumPy:', np.__version__,
      '| scikit-learn:', sklearn.__version__)
check(set(FEATURE_INDICES['D-PHASE']).isdisjoint(FEATURE_INDICES['D-SPREAD']) and
      sorted(FEATURE_INDICES['D-PHASE'] + FEATURE_INDICES['D-SPREAD']) == list(range(54)) and
      [len(FEATURE_INDICES[g]) for g in FAMILIES] == [36, 18, 54],
      'Descriptor-family index definitions inconsistent')

In [ ]:
# 2 — Verify the reusable D-ALL9 reference and saved partitions from Notebook 04
check(REFERENCE_ZIP.is_file(), f'Missing Notebook 04 archive: {REFERENCE_ZIP.name}')
reference_zip_sha = sha256_file(REFERENCE_ZIP)
check(reference_zip_sha == REFERENCE_ZIP_SHA256,
      'Notebook 04 archive fingerprint has changed; stop and reconcile')

with zipfile.ZipFile(REFERENCE_ZIP) as archive:
    reference_summary = json.loads(archive.read('run_summary.json'))
    ref_ledger = from_zip_csv(archive, 'eligible_trial_ledger.csv')
    ref_outer = from_zip_csv(archive, 'outer_fold_manifest.csv')
    ref_inner = from_zip_csv(archive, 'inner_fold_manifest.csv')
    ref_inner_scores = from_zip_csv(archive, 'inner_candidate_scores.csv')
    ref_pooled_scores = from_zip_csv(archive, 'pooled_candidate_scores.csv')
    ref_selected = from_zip_csv(archive, 'selected_models.csv')
    ref_predictions = from_zip_csv(archive, 'held_out_predictions.csv')
    ref_timings = from_zip_csv(archive, 'outer_model_timings.csv')
    ref_checks = from_zip_csv(archive, 'verification_checks.csv')
    ref_metrics = from_zip_csv(archive, 'pooled_metrics.csv')

check(reference_summary['status'] == 'PASS' and
      reference_summary['source_sha256'] == SOURCE_SHA256 and
      reference_summary['total_primary_trials'] == 284,
      'Notebook 04 reference metadata mismatch')
check(ref_checks and all(r['status'] == 'PASS' for r in ref_checks),
      'Notebook 04 saved verification has failures')
check(len(ref_ledger) == 284 and len(ref_outer) == 5 and len(ref_inner) == 20,
      'Notebook 04 trial/fold evidence is incomplete')
ref_by_uid = {r['trial_uid']: r for r in ref_ledger}
check(len(ref_by_uid) == 284 and len({r['subject_id'] for r in ref_ledger}) == 21 and
      sum(int(r['retained_cycles']) for r in ref_ledger) == 60757,
      'Incorrect primary population in Notebook 04 reference')
check(all(np.isfinite(float(r['observed_W_per_kg'])) and
          float(r['observed_W_per_kg']) > 0 for r in ref_ledger),
      'Invalid saved metabolic target')

ref_d_predictions = {r['trial_uid']: r for r in ref_predictions
                     if r['procedure'] == 'D-ALL9_FIXED'}
ref_d_selections = {int(r['outer_fold']): r for r in ref_selected
                    if r['procedure'] == 'D-ALL9_FIXED'}
ref_d_inner = [r for r in ref_inner_scores if r['representation'] == 'D-ALL9']
ref_d_pooled = [r for r in ref_pooled_scores if r['representation'] == 'D-ALL9']
check(len(ref_d_predictions) == 284 and set(ref_d_predictions) == set(ref_by_uid) and
      len(ref_d_selections) == 5 and len(ref_d_inner) == 120 and len(ref_d_pooled) == 30,
      'Notebook 04 D-ALL9 evidence incomplete')
for uid, pred in ref_d_predictions.items():
    record = ref_by_uid[uid]
    check(pred['subject_id'] == record['subject_id'] and
          pred['trial_id'] == record['trial_id'] and
          int(pred['outer_fold']) == int(record['outer_fold']) and
          float(pred['observed_W_per_kg']) == float(record['observed_W_per_kg']) and
          np.isfinite(float(pred['predicted_W_per_kg'])),
          f'Notebook 04 descriptor reference mismatch: {uid}')
for fold in OUTER_FOLDS:
    rows = [r for r in ref_outer if int(r['outer_fold']) == fold]
    check(len(rows) == 1 and set(rows[0]['test_participants'].split(';')) == set(OUTER_FOLDS[fold])
          and set(rows[0]['development_participants'].split(';')) == set(FOLD_OF)-set(OUTER_FOLDS[fold])
          and int(rows[0]['n_test_trials']) == EXPECTED_COUNTS[fold-1],
          f'Saved outer fold {fold} differs from specification')
    candidates = [r for r in ref_d_pooled if int(r['outer_fold']) == fold]
    check(len(candidates) == 6 and set(float(r['alpha']) for r in candidates) == set(ALPHAS),
          f'D-ALL9 reference candidates incomplete for fold {fold}')
    best = min(candidates, key=lambda r: (float(r['inner_pooled_RMSE_W_per_kg']), float(r['alpha'])))
    selected = ref_d_selections[fold]
    check(float(selected['alpha']) == float(best['alpha']) and
          float(selected['inner_pooled_RMSE_W_per_kg']) == float(best['inner_pooled_RMSE_W_per_kg']) and
          int(selected['n_features']) == 54,
          f'D-ALL9 selected configuration mismatch for fold {fold}')

print('Verified Notebook 04 D-ALL9 reference, its six penalties per outer fold, and source provenance')

In [ ]:
# 3 — Reconstruct the frozen 54 descriptors from each source trial
# Descriptor sequence within EACH channel: Q0, Q1, Q2, Q3, phase SD, ROM.
def descriptors_from_trial(trial):
    values = []
    n_cycles = None
    for name in CHANNELS:
        check(name in trial, f'Missing angle channel: {name}')
        angle = np.asarray(trial[name][()], dtype=np.float64)
        check(angle.ndim == 2 and angle.shape[1] == 101 and np.isfinite(angle).all(),
              f'Invalid angle matrix: {name}')
        if n_cycles is None:
            n_cycles = angle.shape[0]
        check(angle.shape[0] == n_cycles, 'Channel cycle-count disagreement')
        mean_curve = angle.mean(axis=0, dtype=np.float64)
        for q in range(4):
            # The two boundary samples are included in adjacent intervals.
            values.append(float(np.trapezoid(mean_curve[25*q:25*q+26], dx=1.0)/25.0))
        values.append(float(np.std(mean_curve, ddof=0)))
        values.append(float(np.max(mean_curve) - np.min(mean_curve)))
    return np.asarray(values, dtype=np.float64), n_cycles


# Lightweight synthetic arithmetic/order check, independent of the source dataset.
class FakeTrial(dict):
    pass

constant = FakeTrial({name: np.full((3, 101), float(j+1))
                      for j, name in enumerate(CHANNELS)})
test_d, test_cycles = descriptors_from_trial(constant)
check(test_cycles == 3 and test_d.shape == (54,), 'Descriptor synthetic dimensions wrong')
for j in range(9):
    expected = np.asarray([j+1.0]*4 + [0.0, 0.0], dtype=np.float64)
    check(np.allclose(test_d[6*j:6*j+6], expected, atol=1e-12, rtol=0),
          f'Unexpected descriptor order in channel {j}')
linear = FakeTrial({name: np.tile(np.linspace(0, 100, 101), (3, 1))
                    for name in CHANNELS})
linear_d, _ = descriptors_from_trial(linear)
check(np.allclose(linear_d[:4], [12.5, 37.5, 62.5, 87.5], atol=1e-12, rtol=0)
      and abs(linear_d[5] - 100.0) < 1e-12,
      'Trapezoidal quartiles or ROM incorrect')

check(H5.is_file(), f'Missing corrected Baček HDF5: {H5.name}')
source_sha = sha256_file(H5)
check(source_sha == SOURCE_SHA256, 'Incorrect HDF5 fingerprint; stop')

records, descriptor_rows = [], []
with h5py.File(H5, 'r') as h5:
    check(set(h5['subjects']) == set(FOLD_OF), 'Unexpected source participant identities')
    for person in sorted(h5['subjects']):
        for trial_id in sorted(h5['subjects'][person]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1 <= int(trial_id[1:]) <= 15):
                continue
            trial = h5['subjects'][person]['trials'][trial_id]
            uid = f'{person}_{trial_id}'
            descriptor, count = descriptors_from_trial(trial)
            for flag in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
                check(flag in trial.attrs and
                      np.asarray(trial.attrs[flag]).reshape(-1)[0] == 1,
                      f'{uid}: source validity flag {flag} is not valid')
            target = np.asarray(trial['metcost_mean'][()])
            check(target.size == 1 and np.issubdtype(target.dtype, np.number) and
                  np.isrealobj(target), f'{uid}: target format invalid')
            observed = float(target.reshape(-1)[0])
            check(np.isfinite(observed) and observed > 0, f'{uid}: target invalid')
            cycle_idx = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(cycle_idx) == count and np.isfinite(cycle_idx).all() and
                  np.all(cycle_idx == np.floor(cycle_idx)) and
                  np.all(np.diff(cycle_idx) > 0), f'{uid}: invalid cycle provenance')
            if uid == 'S06_T11':
                check(count == 85, 'S06_T11 has unexpected cycle count')
                continue
            check(count >= 100 and uid in ref_by_uid, f'{uid}: primary eligibility mismatch')
            ref = ref_by_uid[uid]
            check(ref['subject_id'] == person and ref['trial_id'] == trial_id and
                  int(ref['outer_fold']) == FOLD_OF[person] and
                  int(ref['retained_cycles']) == count and
                  float(ref['observed_W_per_kg']) == observed,
                  f'{uid}: Notebook 04 ledger differs from source')
            records.append({'trial_uid': uid, 'subject_id': person, 'trial_id': trial_id,
                            'outer_fold': FOLD_OF[person], 'retained_cycles': count,
                            'observed_W_per_kg': observed})
            descriptor_rows.append(descriptor)

check(len(records) == 284 and {r['trial_uid'] for r in records} == set(ref_by_uid),
      'Primary trial identities do not match Notebook 04')
check(sum(r['retained_cycles'] for r in records) == 60757 and
      [sum(r['outer_fold'] == fold for r in records) for fold in OUTER_FOLDS] == list(EXPECTED_COUNTS),
      'Primary cycle or outer-fold counts inconsistent')

D = np.stack(descriptor_rows).astype(np.float64, copy=False)
X = {name: D[:, FEATURE_INDICES[name]] for name in FAMILIES}
check(D.shape == (284, 54) and all(X[g].shape == (284, len(FEATURE_INDICES[g])) and
      np.isfinite(X[g]).all() for g in FAMILIES), 'Descriptor matrices invalid')
check(np.array_equal(X['D-FULL'][:, FEATURE_INDICES['D-PHASE']], X['D-PHASE']) and
      np.array_equal(X['D-FULL'][:, FEATURE_INDICES['D-SPREAD']], X['D-SPREAD']),
      'Descriptor-family subsetting incorrect')
y = np.asarray([r['observed_W_per_kg'] for r in records], dtype=np.float64)
groups = np.asarray([r['subject_id'] for r in records])
uids = np.asarray([r['trial_uid'] for r in records])
uid_index = {uid: i for i, uid in enumerate(uids)}
check(len(uid_index) == 284 and set(groups) == set(FOLD_OF), 'Trial indexing invalid')
print('Verified 284 primary trials, 21 participants; descriptor feature dimensions:',
      {g: X[g].shape for g in FAMILIES})

In [ ]:
# 4 — Recover the exact Notebook 04 inner training/validation partitions
inner_by_outer = {}
for fold in OUTER_FOLDS:
    dev_idx = np.flatnonzero(~np.isin(groups, OUTER_FOLDS[fold]))
    test_idx = np.flatnonzero(np.isin(groups, OUTER_FOLDS[fold]))
    test_people = set(OUTER_FOLDS[fold])
    check(len(test_idx) == EXPECTED_COUNTS[fold-1] and
          set(groups[dev_idx]).isdisjoint(test_people),
          f'Outer {fold}: wrong held-out participants')
    saved = sorted([r for r in ref_inner if int(r['outer_fold']) == fold],
                   key=lambda r: int(r['inner_fold']))
    check(len(saved) == 4, f'Outer {fold}: missing saved inner splits')
    partitions, validated = [], []
    for k, row in enumerate(saved, 1):
        train_people = set(row['train_participants'].split(';'))
        val_people = set(row['validation_participants'].split(';'))
        train_idx = np.flatnonzero(np.isin(groups, list(train_people)))
        val_idx = np.flatnonzero(np.isin(groups, list(val_people)))
        check(int(row['inner_fold']) == k and train_people and val_people and
              train_people.isdisjoint(val_people) and
              train_people | val_people == set(groups[dev_idx]) and
              train_people.isdisjoint(test_people) and val_people.isdisjoint(test_people),
              f'Outer {fold}, inner {k}: participant boundary invalid')
        check(len(train_idx) == int(row['n_train_trials']) and
              len(val_idx) == int(row['n_validation_trials']),
              f'Outer {fold}, inner {k}: saved trial counts mismatch')
        partitions.append((train_idx, val_idx))
        validated.extend(val_idx.tolist())
    check(sorted(validated) == sorted(dev_idx.tolist()),
          f'Outer {fold}: inner-validation trials not each covered once')
    inner_by_outer[fold] = (dev_idx, test_idx, partitions)
print('Exact 20 Notebook 04 inner partitions verified and reused')

## Descriptor-family model fitting


In [ ]:
# 5 — Fit D-PHASE and D-SPREAD; import the existing D-FULL procedure

def model_for(alpha):
    return Pipeline([('scale', StandardScaler()),
                     ('ridge', Ridge(alpha=alpha, fit_intercept=True))])


def candidate_key(score):
    # Exact float64 ties within a fixed descriptor family favour smaller alpha.
    return (score['inner_pooled_RMSE_W_per_kg'], score['alpha'])

inner_scores, pooled_scores, selections = [], [], []
new_predictions, model_timings = [], []
started_at = datetime.now(timezone.utc).isoformat()
start = time.perf_counter()
for fold in OUTER_FOLDS:
    dev_idx, test_idx, partitions = inner_by_outer[fold]
    dev_lookup = {int(idx): k for k, idx in enumerate(dev_idx)}
    for family in NEW_FAMILIES:
        matrix = X[family]
        candidate_scores = []
        for alpha in ALPHAS:
            pooled_pred = np.full(len(dev_idx), np.nan, dtype=np.float64)
            for inner, (train_idx, val_idx) in enumerate(partitions, 1):
                model = model_for(alpha)
                t0 = time.perf_counter()
                model.fit(matrix[train_idx], y[train_idx])
                train_seconds = time.perf_counter() - t0
                pred_train = np.asarray(model.predict(matrix[train_idx]), dtype=np.float64)
                pred_val = np.asarray(model.predict(matrix[val_idx]), dtype=np.float64)
                check(np.isfinite(pred_train).all() and np.isfinite(pred_val).all(),
                      f'{family} fold {fold}, inner {inner}: invalid predictions')
                for idx, pred in zip(val_idx, pred_val):
                    pooled_pred[dev_lookup[int(idx)]] = pred
                inner_scores.append({
                    'outer_fold': fold, 'inner_fold': inner, 'family_id': family,
                    'alpha': alpha, 'n_features': matrix.shape[1],
                    'n_train_trials': len(train_idx), 'n_validation_trials': len(val_idx),
                    'train_RMSE_W_per_kg': rmse(y[train_idx], pred_train),
                    'validation_RMSE_W_per_kg': rmse(y[val_idx], pred_val),
                    'fit_seconds': train_seconds, 'source_run': 'NOTEBOOK08',
                })
            check(np.isfinite(pooled_pred).all(), 'Incomplete pooled inner validation')
            score = {'outer_fold': fold, 'family_id': family, 'alpha': alpha,
                     'n_features': matrix.shape[1],
                     'inner_pooled_RMSE_W_per_kg': rmse(y[dev_idx], pooled_pred),
                     'source_run': 'NOTEBOOK08'}
            pooled_scores.append(score)
            candidate_scores.append(score)
        best = min(candidate_scores, key=candidate_key)
        selections.append({'outer_fold': fold, 'family_id': family,
                           'alpha': best['alpha'], 'n_features': best['n_features'],
                           'inner_pooled_RMSE_W_per_kg': best['inner_pooled_RMSE_W_per_kg'],
                           'source_run': 'NOTEBOOK08'})
        final = model_for(best['alpha'])
        t0 = time.perf_counter()
        final.fit(matrix[dev_idx], y[dev_idx])
        fit_seconds = time.perf_counter() - t0
        t0 = time.perf_counter()
        predictions = np.asarray(final.predict(matrix[test_idx]), dtype=np.float64)
        prediction_seconds = time.perf_counter() - t0
        check(len(predictions) == len(test_idx) and np.isfinite(predictions).all(),
              f'{family} fold {fold}: incomplete outer prediction')
        model_timings.append({
            'outer_fold': fold, 'family_id': family, 'alpha': best['alpha'],
            'n_features': best['n_features'], 'fit_seconds': fit_seconds,
            'prediction_seconds': prediction_seconds,
            'prediction_seconds_per_trial': prediction_seconds/len(test_idx),
            'n_train_trials': len(dev_idx), 'n_predicted_trials': len(test_idx),
            'source_run': 'NOTEBOOK08',
        })
        for idx, pred in zip(test_idx, predictions):
            trial = records[idx]
            new_predictions.append({
                'family_id': family, 'outer_fold': fold,
                'subject_id': trial['subject_id'], 'trial_id': trial['trial_id'],
                'trial_uid': trial['trial_uid'],
                'observed_W_per_kg': trial['observed_W_per_kg'],
                'predicted_W_per_kg': float(pred), 'source_run': 'NOTEBOOK08',
            })
        print(f'Fold {fold}: {family:<9s} alpha={best["alpha"]:g}; '
              f'inner RMSE={best["inner_pooled_RMSE_W_per_kg"]:.5f}')

# The full reference has exactly the same 54 descriptors and fitting method as D-ALL9_FIXED.
for row in ref_d_inner:
    inner_scores.append({
        'outer_fold': int(row['outer_fold']), 'inner_fold': int(row['inner_fold']),
        'family_id': 'D-FULL', 'alpha': float(row['alpha']), 'n_features': 54,
        'n_train_trials': int(row['n_train_trials']),
        'n_validation_trials': int(row['n_validation_trials']),
        'train_RMSE_W_per_kg': float(row['train_RMSE_W_per_kg']),
        'validation_RMSE_W_per_kg': float(row['validation_RMSE_W_per_kg']),
        'fit_seconds': float(row['fit_seconds']), 'source_run': 'NOTEBOOK04',
    })
for row in ref_d_pooled:
    pooled_scores.append({
        'outer_fold': int(row['outer_fold']), 'family_id': 'D-FULL',
        'alpha': float(row['alpha']), 'n_features': 54,
        'inner_pooled_RMSE_W_per_kg': float(row['inner_pooled_RMSE_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })
for fold, row in sorted(ref_d_selections.items()):
    selections.append({
        'outer_fold': fold, 'family_id': 'D-FULL',
        'alpha': float(row['alpha']), 'n_features': 54,
        'inner_pooled_RMSE_W_per_kg': float(row['inner_pooled_RMSE_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })
    matches = [r for r in ref_timings
               if int(r['outer_fold']) == fold and r['representation'] == 'D-ALL9' and
               float(r['alpha']) == float(row['alpha'])]
    check(len(matches) == 1, f'Notebook 04 D-ALL9 timing missing in fold {fold}')
    timing = matches[0]
    model_timings.append({
        'outer_fold': fold, 'family_id': 'D-FULL', 'alpha': float(row['alpha']),
        'n_features': 54, 'fit_seconds': float(timing['fit_seconds']),
        'prediction_seconds': float(timing['prediction_seconds']),
        'prediction_seconds_per_trial':
            float(timing['prediction_seconds'])/int(timing['n_predicted_trials']),
        'n_train_trials': int(timing['n_train_trials']),
        'n_predicted_trials': int(timing['n_predicted_trials']),
        'source_run': 'NOTEBOOK04',
    })

elapsed = time.perf_counter()-start
print(f'Two new descriptor families fitted in {elapsed:.2f}s; D-FULL reused from Notebook 04')

In [ ]:
# 6 — Build matched predictions, metrics and participant-wise descriptive comparisons
prediction_ledger = list(new_predictions)
for uid, ref in ref_d_predictions.items():
    prediction_ledger.append({
        'family_id': 'D-FULL', 'outer_fold': int(ref['outer_fold']),
        'subject_id': ref['subject_id'], 'trial_id': ref['trial_id'],
        'trial_uid': uid, 'observed_W_per_kg': float(ref['observed_W_per_kg']),
        'predicted_W_per_kg': float(ref['predicted_W_per_kg']),
        'source_run': 'NOTEBOOK04',
    })

by_family_uid = {(r['family_id'], r['trial_uid']): r for r in prediction_ledger}
check(len(prediction_ledger) == 3*284 and len(by_family_uid) == 3*284,
      'Duplicate or missing descriptor predictions')
check(all({r['trial_uid'] for r in prediction_ledger if r['family_id'] == family}
          == set(uids) for family in FAMILIES),
      'Different descriptor families have different evaluation sets')
for row in prediction_ledger:
    source = records[uid_index[row['trial_uid']]]
    check(row['subject_id'] == source['subject_id'] and
          row['trial_id'] == source['trial_id'] and
          row['outer_fold'] == source['outer_fold'] and
          row['observed_W_per_kg'] == source['observed_W_per_kg'] and
          np.isfinite(row['predicted_W_per_kg']),
          f'Invalid matched prediction: {row["family_id"]} {row["trial_uid"]}')

pooled_metrics, fold_metrics, participant_metrics = [], [], []
for family in FAMILIES:
    ordered = [by_family_uid[(family, uid)] for uid in uids]
    pooled_metrics.append({'family_id': family, **metrics(
        [r['observed_W_per_kg'] for r in ordered],
        [r['predicted_W_per_kg'] for r in ordered])})
    for fold in OUTER_FOLDS:
        subset = [r for r in ordered if r['outer_fold'] == fold]
        fold_metrics.append({'outer_fold': fold, 'family_id': family, **metrics(
            [r['observed_W_per_kg'] for r in subset],
            [r['predicted_W_per_kg'] for r in subset])})
    for subject in sorted(FOLD_OF):
        subset = [r for r in ordered if r['subject_id'] == subject]
        participant_metrics.append({'subject_id': subject, 'family_id': family,
                                    'outer_fold': FOLD_OF[subject], **metrics(
            [r['observed_W_per_kg'] for r in subset],
            [r['predicted_W_per_kg'] for r in subset])})

metric_map = {r['family_id']: r for r in pooled_metrics}
paired_differences = []
participant_differences = []
for family in NEW_FAMILIES:
    paired_differences.append({
        'reference_family': 'D-FULL', 'candidate_family': family,
        'n_matched_trials': 284,
        'RMSE_reference_minus_candidate_W_per_kg':
            metric_map['D-FULL']['RMSE_W_per_kg'] - metric_map[family]['RMSE_W_per_kg'],
        'MAE_reference_minus_candidate_W_per_kg':
            metric_map['D-FULL']['MAE_W_per_kg'] - metric_map[family]['MAE_W_per_kg'],
    })
    for person in sorted(FOLD_OF):
        full = next(r for r in participant_metrics
                    if r['subject_id'] == person and r['family_id'] == 'D-FULL')
        candidate = next(r for r in participant_metrics
                         if r['subject_id'] == person and r['family_id'] == family)
        check(full['n_trials'] == candidate['n_trials'] and full['n_trials'] > 0,
              f'Participant {person}: mismatched trial counts')
        participant_differences.append({
            'subject_id': person, 'candidate_family': family,
            'outer_fold': FOLD_OF[person], 'n_matched_trials': full['n_trials'],
            'RMSE_reference_minus_candidate_W_per_kg':
                full['RMSE_W_per_kg'] - candidate['RMSE_W_per_kg'],
            'MAE_reference_minus_candidate_W_per_kg':
                full['MAE_W_per_kg'] - candidate['MAE_W_per_kg'],
        })

print('\nPooled held-out descriptor-family RMSE (W/kg):')
for row in pooled_metrics:
    print(f'  {row["family_id"]:<9s} {row["RMSE_W_per_kg"]:.6f}')
print('Descriptive participant-wise comparisons saved; clustered uncertainty is evaluated in E5.2.')

In [ ]:
# 7 — Explicit verification evidence for E3.3
checks = []


def verify(name, result):
    checks.append({'check': name, 'status': 'PASS' if bool(result) else 'FAIL'})
    check(result, name)


verify('correct_source_and_notebook04_zip_fingerprints',
       source_sha == SOURCE_SHA256 and reference_zip_sha == REFERENCE_ZIP_SHA256)
verify('primary_284_trial_population_matches_reference',
       len(records) == 284 and set(uids) == set(ref_by_uid) and
       sum(r['retained_cycles'] for r in records) == 60757)
verify('three_frozen_descriptor_dimensions_and_values',
       all(X[g].shape == (284, len(FEATURE_INDICES[g])) and np.isfinite(X[g]).all()
           for g in FAMILIES))
verify('descriptor_ordering_and_quartile_synthetic_tests',
       test_cycles == 3 and np.allclose(linear_d[:4], [12.5, 37.5, 62.5, 87.5],
                                       atol=1e-12, rtol=0))
verify('fixed_outer_fold_counts',
       [sum(r['outer_fold'] == f for r in records) for f in OUTER_FOLDS]
       == list(EXPECTED_COUNTS))
verify('all_20_saved_inner_partitions_reused',
       len(ref_inner) == 20 and len(inner_by_outer) == 5 and
       all(len(inner_by_outer[f][2]) == 4 for f in OUTER_FOLDS))
verify('exactly_360_inner_fit_score_records', len(inner_scores) == 5*3*6*4)
verify('exactly_90_pooled_candidate_scores', len(pooled_scores) == 5*3*6)
verify('exactly_15_selected_models', len(selections) == 5*3)
verify('exactly_10_new_outer_fits_and_568_predictions',
       len(new_predictions) == 2*284 and
       len([r for r in model_timings if r['source_run'] == 'NOTEBOOK08']) == 10)
verify('exactly_852_unique_matched_predictions',
       len(prediction_ledger) == 3*284 and len(by_family_uid) == 3*284)
verify('all_predictions_finite_and_same_trial_targets',
       all(np.isfinite(r['predicted_W_per_kg']) and
           r['observed_W_per_kg'] == records[uid_index[r['trial_uid']]]['observed_W_per_kg']
           for r in prediction_ledger))
verify('exactly_284_predictions_per_family',
       all(sum(r['family_id'] == fam for r in prediction_ledger) == 284 for fam in FAMILIES))
verify('all_15_selections_minimise_inner_rmse', all(
       (r['alpha'], r['inner_pooled_RMSE_W_per_kg']) ==
       (best['alpha'], best['inner_pooled_RMSE_W_per_kg'])
       for r in selections for best in [min(
           [v for v in pooled_scores if v['outer_fold'] == r['outer_fold'] and
            v['family_id'] == r['family_id']], key=candidate_key)]))
verify('d_full_predictions_identical_to_notebook04', all(
       by_family_uid[('D-FULL', uid)]['predicted_W_per_kg'] ==
       float(ref_d_predictions[uid]['predicted_W_per_kg']) for uid in uids))
verify('d_full_selected_penalties_identical_to_notebook04', all(
       r['alpha'] == float(ref_d_selections[r['outer_fold']]['alpha'])
       for r in selections if r['family_id'] == 'D-FULL'))
verify('d_full_metrics_match_notebook04', all(
       abs(float(next(x[field] for x in ref_metrics
                      if x['procedure'] == 'D-ALL9_FIXED')) - metric_map['D-FULL'][field]) < 1e-12
       for field in ('RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2')))
verify('all_metrics_finite', all(
       all(np.isfinite(r[field]) for field in ('RMSE_W_per_kg', 'MAE_W_per_kg',
                                              'bias_W_per_kg', 'R2'))
       for r in pooled_metrics))
verify('participant_metrics_and_matched_differences_complete',
       len(participant_metrics) == 3*21 and len(participant_differences) == 2*21 and
       len(paired_differences) == 2 and
       all(row['n_matched_trials'] == 284 for row in paired_differences))

print('Verification:', len(checks), 'of', len(checks), 'checks passed')

In [ ]:
# 8 — Write evidence files and create the result ZIP (no raw gait arrays)
OUT.mkdir(parents=True, exist_ok=True)

notebook_filename = '08_Bacek_Descriptor_Family_Comparison.ipynb'
code_sha = None
if Path(notebook_filename).is_file():
    notebook_document = json.loads(Path(notebook_filename).read_text(encoding='utf-8'))
    combined_code = ''.join(''.join(c['source']) for c in notebook_document['cells']
                            if c['cell_type'] == 'code')
    code_sha = hashlib.sha256(combined_code.encode('utf-8')).hexdigest()

save_csv(OUT/'eligible_trial_ledger.csv', records,
         ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles','observed_W_per_kg'])
save_csv(OUT/'inner_fold_manifest.csv', ref_inner,
         ['outer_fold','inner_fold','train_participants','validation_participants',
          'outer_test_participants','n_train_trials','n_validation_trials'])
save_csv(OUT/'descriptor_family_definitions.csv', [
    {'family_id': name, 'n_features': len(FEATURE_INDICES[name]),
     'feature_indices_0_based': ';'.join(str(i) for i in FEATURE_INDICES[name]),
     'channel_order': ';'.join(CHANNELS),
     'descriptor_order_per_channel': 'Q0;Q1;Q2;Q3;phase_SD;ROM',
     'feature_basis': 'trial-mean waveform; 101 phase samples',
     'estimator': 'StandardScaler -> Ridge'} for name in FAMILIES],
    ['family_id','n_features','feature_indices_0_based','channel_order',
     'descriptor_order_per_channel','feature_basis','estimator'])
save_csv(OUT/'inner_candidate_scores.csv', inner_scores,
         ['outer_fold','inner_fold','family_id','alpha','n_features','n_train_trials',
          'n_validation_trials','train_RMSE_W_per_kg','validation_RMSE_W_per_kg',
          'fit_seconds','source_run'])
save_csv(OUT/'pooled_candidate_scores.csv', pooled_scores,
         ['outer_fold','family_id','alpha','n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'selected_models.csv', selections,
         ['outer_fold','family_id','alpha','n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'held_out_predictions.csv', prediction_ledger,
         ['family_id','outer_fold','subject_id','trial_id','trial_uid',
          'observed_W_per_kg','predicted_W_per_kg','source_run'])
save_csv(OUT/'pooled_metrics.csv', pooled_metrics,
         ['family_id','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'outer_fold_metrics.csv', fold_metrics,
         ['outer_fold','family_id','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'participant_metrics.csv', participant_metrics,
         ['subject_id','family_id','outer_fold','n_trials','RMSE_W_per_kg',
          'MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'paired_rmse_differences.csv', paired_differences,
         ['reference_family','candidate_family','n_matched_trials',
          'RMSE_reference_minus_candidate_W_per_kg','MAE_reference_minus_candidate_W_per_kg'])
save_csv(OUT/'participant_paired_differences.csv', participant_differences,
         ['subject_id','candidate_family','outer_fold','n_matched_trials',
          'RMSE_reference_minus_candidate_W_per_kg','MAE_reference_minus_candidate_W_per_kg'])
save_csv(OUT/'outer_model_timings.csv', model_timings,
         ['outer_fold','family_id','alpha','n_features','fit_seconds','prediction_seconds',
          'prediction_seconds_per_trial','n_train_trials','n_predicted_trials','source_run'])
save_csv(OUT/'verification_checks.csv', checks, ['check','status'])

summary = {
    'status': 'PASS' if all(r['status'] == 'PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment': 'E3.3',
    'source_hdf5': H5.name, 'source_sha256': source_sha,
    'notebook04_results_filename': REFERENCE_ZIP.name,
    'notebook04_results_zip_sha256': reference_zip_sha,
    'notebook04_code_sha256': reference_summary['notebook_code_sha256'],
    'population': {'trials':284,'participants':21,'retained_cycles':60757},
    'target': 'stored corrected net metabolic metcost_mean; W/kg',
    'evaluation': 'internal new-person held-out; previously inspected fixed five outer folds',
    'inner_partitions': 'exact Notebook 04 saved four participant-grouped inner folds per outer fold',
    'families': {name: list(FEATURE_INDICES[name]) for name in FAMILIES},
    'alpha_grid': list(ALPHAS),
    'pipeline': 'trial mean -> fixed descriptor family -> training-only StandardScaler -> Ridge',
    'selection': 'pooled inner trial-weighted RMSE; exact ties favour smaller alpha',
    'reused_procedure': 'D-FULL reuses complete D-ALL9_FIXED evidence from Notebook 04',
    'bootstrap_included': False,
    'n_new_predictions': len(new_predictions),
    'n_matched_predictions': len(prediction_ledger),
    'started_at_utc': started_at,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'new_model_execution_seconds': elapsed,
    'software': {'python': platform.python_version(), 'numpy': np.__version__,
                 'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
                 'h5py': h5py.__version__, 'os': platform.platform(),
                 'cpu_count': os.cpu_count(), 'kernel_python_executable': sys.executable},
    'notebook_filename': notebook_filename,
    'notebook_code_sha256': code_sha,
    'code_fingerprint_basis': 'concatenated code-cell sources, UTF-8',
    'verification': {row['check']: row['status'] for row in checks},
}
(OUT/'run_summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')

files = (
    'eligible_trial_ledger.csv', 'inner_fold_manifest.csv', 'descriptor_family_definitions.csv',
    'inner_candidate_scores.csv', 'pooled_candidate_scores.csv', 'selected_models.csv',
    'held_out_predictions.csv', 'pooled_metrics.csv', 'outer_fold_metrics.csv',
    'participant_metrics.csv', 'paired_rmse_differences.csv', 'participant_paired_differences.csv',
    'outer_model_timings.csv', 'verification_checks.csv', 'run_summary.json',
)
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as dest:
    for name in files:
        dest.write(OUT/name, arcname=name)
print('Saved:', ARCHIVE, '| evidence files:', len(files))
print('Final verification:', summary['status'])

## Comparison interpretation
